# Canonical quantum amplitude estimation

This minimal example estimates the good-state probability $a=\sin^2(\pi/8)$ with three estimation qubits. The value is exactly representable because the amplification eigenphases lie on the $1/8$ QPE grid. The construction follows Brassard, Høyer, Mosca, and Tapp, [*Quantum Amplitude Amplification and Estimation*](https://arxiv.org/abs/quant-ph/0005055).

In [ ]:
import numpy as np

import qarp
from qarp.algorithms import AmplitudeEstimation
from qarp.algorithms import Sampler
from qarp.blocks import PhaseShiftBlock, SimpleBlock

## State preparation and phase oracle

$R_y(2\theta)|0\rangle=\cos(\theta)|0\rangle+\sin(\theta)|1\rangle$. We mark $|1\rangle$ with the exact oracle $O_{\mathrm{good}}=\operatorname{diag}(1,-1)=I-2|1\rangle\langle1|$.

In [ ]:
theta = np.pi / 8
expected_amplitude = np.sin(theta) ** 2

state_preparation = SimpleBlock(1, name="A")
state_preparation.ry(0, 2 * theta)
oracle = PhaseShiftBlock(np.pi, name="O_good")

## Run QAE

The sampler measures only the three-qubit estimation register. Its keys are LSB-first. Raw conjugate peaks at integer labels 1 and 7 are folded into phase bin 1 before converting the phase to an amplitude.

In [ ]:
algorithm = AmplitudeEstimation(
    state_preparation,
    oracle,
    n_ancilla=3,
    primitive=Sampler(n_shots=qarp.EXACT),
).build()
estimate = algorithm.run()

assert np.isclose(estimate, expected_amplitude, atol=1e-12)
assert algorithm.phase_bin == 1
assert np.isclose(algorithm.result_probability, 1.0, atol=1e-12)

print("Raw QPE distribution:", algorithm.distribution)
print("Folded distribution:", algorithm.folded_distribution)
print("Estimated amplitude:", estimate)
print("Analytic amplitude:", expected_amplitude)